In [ ]:
# CELL 1: 필수 패키지 설치
!pip -q install "qwen-tts==0.1.1" "transformers==4.57.3" "accelerate==1.12.0" \
                 "huggingface_hub>=0.34.0,<1.0" bitsandbytes pandas soundfile librosa psutil tqdm hqq



In [ ]:
# Flash attn 패키지 설치 중 CPU 코어 많이써서 Ram 터지는 현상 발생
# 실험2. L4 + BF16(bfloat16) + FlashAttention-2(플래시어텐션2)용 안전하게 2코어만 사용
import os
os.environ["MAX_JOBS"] = "2"


In [ ]:
# 2) 기본 빌드
!pip install -U pip setuptools wheel packaging ninja psutil
!ninja --version


1.13.0.git.kitware.jobserver-pipe-1


In [ ]:
# flash-attn install
!pip uninstall -y flash-attn
!pip install -v flash-attn --no-build-isolation

Using pip 26.0.1 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 29.8 MB/s  0:00:00
  Running command Preparing metadata (pyproject.toml)
  /usr/local/lib/python3.12/dist-packages/wheel/bdist_wheel.py:4: FutureWarning: The 'wheel' package is no longer the canonical location of the 'bdist_wheel' command, and will be removed in a future release. Please update to setuptools v70.1 or later which contains an integrated version of this command.
    warn(
  /usr/local/lib/python3.12/dist-packages/setuptools/dist.py:765: SetuptoolsDeprecationWarning: License classifiers are deprecated.
  !!

          ********************************************************************************
          Please consider removing the following classifiers in favor of a SPDX license expression:

          License :: OSI Approved :: BSD License

          See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for

In [ ]:
# CELL 2: Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

Mounted at /content/drive


In [ ]:
# CELL 3: Worker -> 2 colab 진행
# Session A는 WORKER_INDEX=0, Session B는 WORKER_INDEX=1 로 바꾸세요.

WORKER_INDEX = 2
WORKER_COUNT = 1
WORKER_TAG = f"colab{WORKER_INDEX}"

# Project path
PROJECT_ROOT = "/content/drive/MyDrive/Colab Notebooks/Qwen3-TTS_Quantization_lowbit"

# DB path
DATA_ROOT = "/content/drive/MyDrive/Colab Notebooks/Data/"

# Hugging Face cache
CACHE_DIR = "/content/hf_cache"

# Debug enable
ENABLE_CUDA_LAUNCH_BLOCKING = False

if ENABLE_CUDA_LAUNCH_BLOCKING:
    import os
    os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

print("WORKER_INDEX :", WORKER_INDEX)
print("WORKER_COUNT :", WORKER_COUNT)
print("WORKER_TAG   :", WORKER_TAG)
print("PROJECT_ROOT :", PROJECT_ROOT)
print("CACHE_DIR    :", CACHE_DIR)

WORKER_INDEX : 2
WORKER_COUNT : 1
WORKER_TAG   : colab2
PROJECT_ROOT : /content/drive/MyDrive/Colab Notebooks/Qwen3-TTS_Quantization_lowbit
CACHE_DIR    : /content/hf_cache


In [ ]:
# CELL 4: 유틸

# 1차실험 -> GPU : T4로 진행 -> BF16 대신 FP32 사용
# 2차실험 -> GPU : L4로 진행 -> FlashAtt_2 , BF16 사용
from __future__ import annotations

import datetime as dt
import gc
import json
import os
import platform
import random
import re
import shutil
import sys
import time
import traceback
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional

import librosa
import numpy as np
import pandas as pd
import psutil
import soundfile as sf
import torch
import torch.nn as nn
from tqdm.auto import tqdm

# Quantization import

# 8bit, 4bit    -> bitsandbytes사용
try:
    import bitsandbytes as bnb
except Exception:
    bnb = None

# 3bit, 2bit, 1bit(생략) -> HQQ 사용
try:
    from hqq.core.quantize import HQQLinear, BaseQuantizeConfig
except Exception:
    HQQLinear = None
    BaseQuantizeConfig = None

#transformer import
try:
    import transformers
except Exception:
    transformers = None

from huggingface_hub import snapshot_download
from qwen_tts import Qwen3TTSModel

_MB = 1024 ** 2


def now_iso() -> str:
    return dt.datetime.now().isoformat(timespec="seconds")


def cleanup_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.reset_peak_memory_stats()
        except Exception:
            pass


def ensure_mono_float32(audio: np.ndarray) -> np.ndarray:
    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    return audio.astype(np.float32, copy=False)


def dtype_to_str(dtype: object) -> str:
    if dtype is None:
        return "unknown"
    return str(dtype).replace("torch.", "")


def pick_base_dtype() -> torch.dtype:
    # L4 기준으로 BF16(bfloat16)
    return torch.bfloat16


def validate_runtime_for_flash_bf16(expected_gpu_substr: str = "L4") -> None:
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU가 필요합니다. Colab 런타임을 GPU(L4)로 바꿔주세요.")

    gpu_name = torch.cuda.get_device_name(0)
    major, minor = torch.cuda.get_device_capability(0)

    print(f"gpu_name      : {gpu_name}")
    print(f"gpu_capability: {major}.{minor}")
    print(f"bf16_supported: {torch.cuda.is_bf16_supported()}")

    if expected_gpu_substr and expected_gpu_substr not in gpu_name:
        print(f"[warn] 현재 GPU는 {gpu_name} 입니다. 요청한 GPU는 {expected_gpu_substr} 기준입니다.")

    if major < 8:
        raise RuntimeError(     # T4로는 진행 불가 -> 1차실험 후 수정
            f"FlashAttention-2(플래시어텐션2) + BF16(bfloat16) 조합은 Ampere 이상 GPU가 필요합니다. 현재 capability={major}.{minor}"
        )

    if not torch.cuda.is_bf16_supported():
        raise RuntimeError("현재 GPU/런타임은 BF16(bfloat16)을 지원하지 않습니다.")


def setup_global_runtime(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

    torch.set_grad_enabled(False)


@dataclass(frozen=True)
class ModelConfig:
    model_id: str
    model_tag: str


@dataclass(frozen=True)
class RunPlan:
    name: str
    llm_bit: Optional[int]
    mtp_bit: Optional[int]


@dataclass
class ExperimentConfig:
    project_root: Path
    data_root: Path
    audio_ref_dir: Path
    transcription_dir: Path
    manifest_dir: Path
    gen_dir: Path
    result_dir: Path
    cache_dir: Path
    log_dir: Path
    reference_manifest_csv: Path

    worker_tag: str
    worker_index: int
    worker_count: int
    runtime_env: str

    seed: int
    base_dtype: torch.dtype
    attn_implementation: str
    target_device: str
    max_new_tokens: int
    warmup_count: int
    temperature: float
    top_p: float
    top_k: int
    do_sample: bool
    non_streaming_mode: bool
    subtalker_dosample: bool
    use_cache: bool
    batch_size: int

    preload_model_snapshots: bool
    run_preflight_smoke: bool
    keep_lm_head_in_base_dtype: bool
    resume_incomplete_runs: bool
    write_partial_csv_every_n: int

    model_configs: list[ModelConfig] = field(default_factory=list)
    run_plans_all: list[RunPlan] = field(default_factory=list)
    run_plans: list[RunPlan] = field(default_factory=list)

    @property
    def base_dtype_str(self) -> str:
        return dtype_to_str(self.base_dtype)

    @property
    def sampling_config(self) -> dict[str, object]:
        return {
            "temperature": self.temperature,
            "top_p": self.top_p,
            "top_k": self.top_k,
            "max_new_tokens": self.max_new_tokens,
            "do_sample": self.do_sample,
            "non_streaming_mode": self.non_streaming_mode,
            "subtalker_dosample": self.subtalker_dosample,
            "use_cache": self.use_cache,
            "batch_size": self.batch_size,
        }

# Lowbit 측정
def default_model_configs() -> list[ModelConfig]:
    return [
        ModelConfig("Qwen/Qwen3-TTS-12Hz-1.7B-Base", "1.7B"),
    ]

# RunPlan : worker 수마다 분할 -> 1대면 전체스캔, 2대면 짝 홀
def default_run_plans() -> list[RunPlan]:
    return [
        RunPlan("bf16_baseline", None, None),
        RunPlan("llm8_mtp8", 8, 8),
        RunPlan("llm4_mtp8", 4, 8),
        RunPlan("llm8_mtp4", 8, 4),
        RunPlan("llm4_mtp4", 4, 4),
        RunPlan("llm4_mtp3", 4, 3),
        RunPlan("llm4_mtp2", 4, 2),
        RunPlan("llm3_mtp4", 3, 4),
        RunPlan("llm2_mtp4", 2, 4),
        RunPlan("llm3_mtp3", 3, 3),
        RunPlan("llm2_mtp2", 2, 2),
    ]


def split_run_plans(plans: list[RunPlan], worker_index: int, worker_count: int) -> list[RunPlan]:
    if worker_count <= 1:
        return plans
    return [plan for idx, plan in enumerate(plans) if idx % worker_count == worker_index]


def build_config() -> ExperimentConfig:
    project_root = Path(PROJECT_ROOT)
    data_root = Path(DATA_ROOT) / "Qwen3-TTS_DATA"          # Google Drive path -> DATA path와 Code path따로 두기.
    gen_dir = project_root / "generations" / WORKER_TAG
    result_dir = project_root / "results" / WORKER_TAG
    log_dir = project_root / "logs" / WORKER_TAG
    cache_dir = Path(CACHE_DIR)

    validate_runtime_for_flash_bf16(expected_gpu_substr="L4")

    cfg = ExperimentConfig(
        project_root=project_root,
        data_root=data_root,
        audio_ref_dir=data_root / "Audio" / "reference",
        transcription_dir=data_root / "Transcription",
        manifest_dir=data_root / "Manifests",
        gen_dir=gen_dir,
        result_dir=result_dir,
        cache_dir=cache_dir,
        log_dir=log_dir,
        reference_manifest_csv=data_root / "Manifests" / "reference_manifest.csv",
        worker_tag=WORKER_TAG,
        worker_index=WORKER_INDEX,
        worker_count=WORKER_COUNT,
        runtime_env="colab",
        seed=66,
        base_dtype=pick_base_dtype(),
        attn_implementation="flash_attention_2",
        target_device="cuda" if torch.cuda.is_available() else "cpu",

        max_new_tokens=512,                                                       # 1차 실험 2048로 진행(기존 Qwen3-TTS와 동일) -> token prediction 할 때 eos토큰을 못뱉어서 길어지는 현상 발생 -> WER, CER 기하급수적 증가
                                                                                  # 2차 512로 설정
        warmup_count=1,
        temperature=1.0,
        top_p=0.95,
        top_k=50,
        do_sample=True,
        non_streaming_mode=True,
        subtalker_dosample=False,
        use_cache=True,
        batch_size=8,                                                             # batch size는 메모리 상태보고 유동적으로 변경
        preload_model_snapshots=True,
        run_preflight_smoke=False,
        keep_lm_head_in_base_dtype=True,
        resume_incomplete_runs=True,
        write_partial_csv_every_n=10,                                             # 10개마다 기록
        model_configs=default_model_configs(),
        run_plans_all=default_run_plans(),
    )
    cfg.run_plans = split_run_plans(cfg.run_plans_all, cfg.worker_index, cfg.worker_count)

    for path in [
        cfg.project_root, cfg.data_root, cfg.manifest_dir,
        cfg.gen_dir, cfg.result_dir, cfg.cache_dir, cfg.log_dir
    ]:
        path.mkdir(parents=True, exist_ok=True)

    setup_global_runtime(cfg.seed)
    return cfg


cfg = build_config()
print("runtime_env   :", cfg.runtime_env)
print("target_device :", cfg.target_device)
print("base_dtype    :", cfg.base_dtype_str)
print("attn_impl     :", cfg.attn_implementation)
print("worker_tag    :", cfg.worker_tag)
print("run_plans     :", [p.name for p in cfg.run_plans])


    If you do not have SoX, proceed here:
     - - - http://sox.sourceforge.net/ - - -

    If you do (or think that you should) have SoX, double-check your
    path variables.
    


gpu_name      : NVIDIA L4
gpu_capability: 8.9
bf16_supported: True
runtime_env   : colab
target_device : cuda
base_dtype    : bfloat16
attn_impl     : flash_attention_2
worker_tag    : colab2
run_plans     : ['bf16_baseline', 'llm8_mtp8', 'llm4_mtp8', 'llm8_mtp4', 'llm4_mtp4', 'llm4_mtp3', 'llm4_mtp2', 'llm3_mtp4', 'llm2_mtp4', 'llm3_mtp3', 'llm2_mtp2']


In [ ]:
# CELL 5: selective quantization 유틸
LM_BACKBONE_PREFIXES = ("talker.model",)
LM_HEAD_PREFIXES = ("talker.codec_head", "talker.text_projection")
MTP_PREFIXES = ("talker.code_predictor",)


def has_prefix(name: str, prefixes: tuple[str, ...]) -> bool:
    return any(name.startswith(prefix) for prefix in prefixes)


def detect_module_group(name: str) -> str:
    if has_prefix(name, LM_HEAD_PREFIXES):
        return "lm_head"
    if has_prefix(name, LM_BACKBONE_PREFIXES):
        return "lm_backbone"
    if has_prefix(name, MTP_PREFIXES):
        return "mtp"
    return "other"


def linear_param_count(module: nn.Module) -> int:
    total = 0
    for param in module.parameters(recurse=False):
        total += int(param.numel())
    return total


def tensor_dtype_str(tensor) -> str:
    try:
        return dtype_to_str(tensor.dtype)
    except Exception:
        return "unknown"


def choose_hqq_group_size(linear: nn.Linear, nbits: int) -> int:
    # HQQ 공식 가이드는 4bit에서 group_size=64, axis=1을 추천.
    # 낮은 bit에서는 조금 더 작은 group_size가 안정적인 편이라 32를 우선 사용.
    preferred = [64, 32, 16, 8] if nbits >= 4 else [32, 16, 8]
    numel = int(linear.weight.numel())
    for g in preferred:
        if numel % g == 0:
            return g
    return numel  # 마지막 fallback


def make_int8_layer(linear: nn.Linear):
    new_mod = bnb.nn.Linear8bitLt(
        linear.in_features,
        linear.out_features,
        bias=(linear.bias is not None),
        has_fp16_weights=False,
        threshold=6.0,
    )
    new_mod.weight = bnb.nn.Int8Params(
        linear.weight.detach().cpu().clone(),
        requires_grad=False,
        has_fp16_weights=False,
    )
    if linear.bias is not None:
        new_mod.bias = nn.Parameter(linear.bias.detach().cpu().clone(), requires_grad=False)
    return new_mod


def make_int4_layer(linear: nn.Linear, compute_dtype: torch.dtype):
    new_mod = bnb.nn.Linear4bit(
        linear.in_features,
        linear.out_features,
        bias=(linear.bias is not None),
        compute_dtype=compute_dtype,
        compress_statistics=True,
        quant_type="nf4",
    )
    new_mod.weight = bnb.nn.Params4bit(
        linear.weight.detach().cpu().clone(),
        requires_grad=False,
        quant_type="nf4",
        compress_statistics=True,
    )
    if linear.bias is not None:
        new_mod.bias = nn.Parameter(linear.bias.detach().cpu().clone(), requires_grad=False)
    return new_mod


def make_hqq_layer(
    linear: nn.Linear,
    nbits: int,
    compute_dtype: torch.dtype,
    quant_device: str,
):
    if HQQLinear is None or BaseQuantizeConfig is None:
        raise ImportError("hqq가 설치되지 않았습니다.")

    # 이후 실험에서 사용할 QAT -> retraining 필요하기 때문
    if abs(float(nbits) - 1.58) < 1e-6:
        raise ValueError(
            "1.58bit는 HQQ PTQ로 바로 바꾸는 방식이 아니라 BitNet 계열 QAT/재학습이 필요합니다."
        )

    if nbits not in (1, 2, 3):
        raise ValueError(f"HQQ 경로는 1/2/3bit만 처리합니다. 받은 값: {nbits}")

    group_size = choose_hqq_group_size(linear, nbits)
    quant_config = BaseQuantizeConfig(
        nbits=nbits,
        group_size=group_size,
        axis=1,
    )

    # 모델을 CPU로 로드한 뒤 선택적으로 양자화 후 GPU로 옮겨서 진행하기
    # 각 linear 레이어만 잠깐 quant_device로 옮겨서 양자화.
    work_linear = linear.to(quant_device)

    new_mod = HQQLinear(
        work_linear,
        quant_config=quant_config,
        compute_dtype=compute_dtype,
        device=quant_device,
        initialize=True,
        del_orig=True,
    )

    # bias는 requires_grad=False로 고정
    if getattr(new_mod, "bias", None) is not None:
        new_mod.bias = nn.Parameter(new_mod.bias.detach(), requires_grad=False)

    return new_mod


def get_new_weight_dtype_repr(new_layer: nn.Module) -> str:
    weight = getattr(new_layer, "weight", None)
    if weight is not None:
        return tensor_dtype_str(weight)

    cls_name = type(new_layer).__name__.lower()
    if "hqq" in cls_name:
        return "hqq_packed"
    return "unknown"


def make_inventory_row(
    name: str,
    module: nn.Linear,
    module_group: str,
    llm_bit: Optional[int],
    mtp_bit: Optional[int],
    compute_dtype: torch.dtype,
) -> dict[str, object]:
    row = {
        "module_name": name,
        "module_group": module_group,
        "original_class": type(module).__name__,
        "new_class": type(module).__name__,
        "in_features": int(getattr(module, "in_features", -1)),
        "out_features": int(getattr(module, "out_features", -1)),
        "param_count": linear_param_count(module),
        "original_weight_dtype": tensor_dtype_str(getattr(module, "weight", None)),
        "new_weight_dtype": tensor_dtype_str(getattr(module, "weight", None)),
        "assigned_bit_width": "",
        "quantized": 0,
        "skipped_reason": "",
        "assigned_quant_method": "none",
        "llm_bit": llm_bit if llm_bit is not None else "",
        "mtp_bit": mtp_bit if mtp_bit is not None else "",
        "compute_dtype": dtype_to_str(compute_dtype),
    }
    return row


def summarize_inventory(inventory: list[dict[str, object]]) -> dict[str, int]:
    summary = {
        "n_linear_total": len(inventory),
        "n_quantized_lm_backbone": 0,
        "n_quantized_mtp": 0,
        "n_quantized_lm_head": 0,
        "n_skipped_lm_backbone": 0,
        "n_skipped_mtp": 0,
        "n_skipped_lm_head": 0,
        "n_skipped_other": 0,
    }
    for row in inventory:
        group = str(row.get("module_group", "other"))
        quantized = int(row.get("quantized", 0) or 0)
        if quantized:
            if group == "lm_backbone":
                summary["n_quantized_lm_backbone"] += 1
            elif group == "mtp":
                summary["n_quantized_mtp"] += 1
            elif group == "lm_head":
                summary["n_quantized_lm_head"] += 1
        else:
            if group == "lm_backbone":
                summary["n_skipped_lm_backbone"] += 1
            elif group == "mtp":
                summary["n_skipped_mtp"] += 1
            elif group == "lm_head":
                summary["n_skipped_lm_head"] += 1
            else:
                summary["n_skipped_other"] += 1
    return summary


def apply_selective_quantization(
    model: nn.Module,
    llm_bit: Optional[int],
    mtp_bit: Optional[int],
    compute_dtype: torch.dtype = torch.float16,
    keep_lm_head_in_base_dtype: bool = True,
    quant_device: str = "cuda",
) -> dict[str, object]:
    named_modules_list = list(model.named_modules())
    inventory: list[dict[str, object]] = []

    if llm_bit is None and mtp_bit is None:
        for name, module in named_modules_list:
            if not isinstance(module, nn.Linear):
                continue
            module_group = detect_module_group(name)
            row = make_inventory_row(
                name=name,
                module=module,
                module_group=module_group,
                llm_bit=llm_bit,
                mtp_bit=mtp_bit,
                compute_dtype=compute_dtype,
            )
            row["assigned_bit_width"] = "fp"
            row["skipped_reason"] = "baseline_no_quantization"
            inventory.append(row)

        inventory_summary = summarize_inventory(inventory)
        print(f"  [quant] baseline — 양자화 없음 (linear={len(inventory)})")
        return {
            "n_quantized": 0,
            "n_skipped": len(inventory),
            "inventory": inventory,
            **inventory_summary,
        }

# quantization Exception
    if any(bit in (8, 4) for bit in (llm_bit, mtp_bit)) and bnb is None:
        raise ImportError("bitsandbytes가 설치되지 않았습니다.")

    if any(bit in (3, 2, 1) for bit in (llm_bit, mtp_bit)) and (HQQLinear is None or BaseQuantizeConfig is None):
        raise ImportError("hqq가 설치되지 않았습니다.")

    if any((bit is not None and abs(float(bit) - 1.58) < 1e-6) for bit in (llm_bit, mtp_bit)):
        raise ValueError("1.58bit는 이 노트북 방식의 PTQ로 지원하지 않습니다.")

    n_quantized, n_skipped = 0, 0

    for name, module in named_modules_list:
        if not isinstance(module, nn.Linear):
            continue

        module_group = detect_module_group(name)
        row = make_inventory_row(
            name=name,
            module=module,
            module_group=module_group,
            llm_bit=llm_bit,
            mtp_bit=mtp_bit,
            compute_dtype=compute_dtype,
        )

        is_lm_backbone = module_group == "lm_backbone"
        is_mtp = module_group == "mtp"
        is_lm_head = module_group == "lm_head"

        target_bit = None
        if is_lm_backbone:
            target_bit = llm_bit
        elif is_mtp:
            target_bit = mtp_bit

        row["assigned_bit_width"] = target_bit if target_bit is not None else "fp"

        if is_lm_head and keep_lm_head_in_base_dtype:
            n_skipped += 1
            row["skipped_reason"] = "kept_in_base_dtype"
            inventory.append(row)
            continue

        if target_bit is None:
            n_skipped += 1
            row["skipped_reason"] = "no_target_bit"
            inventory.append(row)
            continue

        parts = name.split(".")
        parent = model
        for part in parts[:-1]:
            parent = getattr(parent, part)
        attr = parts[-1]

        if target_bit == 8:
            new_layer = make_int8_layer(module)
            row["assigned_quant_method"] = "bnb_int8"
        elif target_bit == 4:
            new_layer = make_int4_layer(module, compute_dtype)
            row["assigned_quant_method"] = "bnb_nf4"
        elif target_bit in (3, 2, 1):
            new_layer = make_hqq_layer(
                module,
                nbits=target_bit,
                compute_dtype=compute_dtype,
                quant_device=quant_device,
            )
            row["assigned_quant_method"] = f"hqq_{target_bit}bit"
        else:
            raise ValueError(f"지원하지 않는 bit 수: {target_bit}")

        setattr(parent, attr, new_layer)

        row["quantized"] = 1
        row["new_class"] = type(new_layer).__name__
        row["new_weight_dtype"] = get_new_weight_dtype_repr(new_layer)
        row["skipped_reason"] = ""
        inventory.append(row)
        n_quantized += 1

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    inventory_summary = summarize_inventory(inventory)
    print(f"  [quant] 완료 — quantized: {n_quantized}, skipped: {n_skipped}, linear={len(inventory)}")
    return {
        "n_quantized": n_quantized,
        "n_skipped": n_skipped,
        "inventory": inventory,
        **inventory_summary,
    }

In [ ]:
# CELL 6: 모델 로드 + snapshot(스냅샷) + 메모리 정보
_LOCAL_MODEL_CACHE: dict[str, Path] = {}


def directory_size_mb(path: Path) -> float:
    total = 0
    for item in path.rglob("*"):
        if item.is_file():
            try:
                total += item.stat().st_size
            except OSError:
                pass
    return round(total / _MB, 2)


def get_model_memory_footprint_mb(model) -> float:
    try:
        if hasattr(model, "get_memory_footprint"):
            value = model.get_memory_footprint()
            if isinstance(value, (int, float)):
                return round(float(value) / _MB, 2)
    except Exception:
        pass

    total_bytes = 0
    seen_ptrs: set[int] = set()
    for tensor in list(model.parameters()) + list(model.buffers()):
        try:
            storage = tensor.untyped_storage()
            ptr = storage.data_ptr()
            if ptr in seen_ptrs:
                continue
            seen_ptrs.add(ptr)
            total_bytes += storage.nbytes()
        except Exception:
            total_bytes += tensor.nelement() * tensor.element_size()
    return round(total_bytes / _MB, 2)

def describe_quantization(llm_bit: Optional[int], mtp_bit: Optional[int]) -> dict[str, object]:
    if llm_bit is None and mtp_bit is None:
        return {
            "quant_method": "none",
            "bit_width": "fp",
            "llm_quant_method": "none",
            "mtp_quant_method": "none",
        }

    def _method(bit: Optional[int]) -> str:
        if bit is None:
            return "none"
        if bit == 8:
            return "bnb_int8"
        if bit == 4:
            return "bnb_nf4"
        if bit in (3, 2, 1):
            return f"hqq_{bit}bit"
        return f"custom_{bit}bit"

    return {
        "quant_method": "selective_mixed_precision",
        "bit_width": f"llm{llm_bit if llm_bit is not None else 'fp'}_mtp{mtp_bit if mtp_bit is not None else 'fp'}",
        "llm_quant_method": _method(llm_bit),
        "mtp_quant_method": _method(mtp_bit),
    }

def resolve_local_model_path(model_id: str, cfg: ExperimentConfig) -> Path:
    if model_id in _LOCAL_MODEL_CACHE:
        return _LOCAL_MODEL_CACHE[model_id]

    local_path = snapshot_download(
        repo_id=model_id,
        resume_download=True,
        cache_dir=str(cfg.cache_dir),
    )
    root_dir = Path(local_path)
    speech_dir = root_dir / "speech_tokenizer"
    speech_dir.mkdir(parents=True, exist_ok=True)

    for fname in ["preprocessor_config.json", "config.json"]:
        src = root_dir / fname
        dst = speech_dir / fname
        if not dst.exists() and src.exists():
            shutil.copy2(src, dst)
            print(f"  [fix] copied {fname} -> speech_tokenizer/")

    _LOCAL_MODEL_CACHE[model_id] = root_dir
    return root_dir


def preload_model_snapshots(cfg: ExperimentConfig) -> None:
    if not cfg.preload_model_snapshots:
        return
    print("\n[preload] HF snapshot 캐시 확보 중...")
    for model_cfg in cfg.model_configs:
        print(f"  {model_cfg.model_id}")
        resolve_local_model_path(model_cfg.model_id, cfg)
    print("[preload] 완료")


def load_model_for_run(
    cfg: ExperimentConfig,
    model_id: str,
    llm_bit: Optional[int],
    mtp_bit: Optional[int],
):
    load_t0 = time.perf_counter()
    compute_dtype = cfg.base_dtype
    root_dir = resolve_local_model_path(model_id, cfg)

    tts_wrapper = Qwen3TTSModel.from_pretrained(
        str(root_dir),
        device_map="cpu",
        dtype=compute_dtype,
        attn_implementation=cfg.attn_implementation,
        trust_remote_code=True,
    )

    quant_report = apply_selective_quantization(
        tts_wrapper.model,
        llm_bit=llm_bit,
        mtp_bit=mtp_bit,
        compute_dtype=compute_dtype,
        keep_lm_head_in_base_dtype=cfg.keep_lm_head_in_base_dtype,
        quant_device=cfg.target_device if torch.cuda.is_available() else "cpu",
    )

    tts_wrapper.model = tts_wrapper.model.to(cfg.target_device)

    try:
        if hasattr(tts_wrapper.model, "config"):
            setattr(tts_wrapper.model.config, "_attn_implementation", cfg.attn_implementation)
    except Exception:
        pass

    tts_wrapper.model.eval()

    try:
        tts_wrapper.device = next(tts_wrapper.model.parameters()).device
    except StopIteration:
        tts_wrapper.device = torch.device(cfg.target_device)

    model_memory_footprint_mb = get_model_memory_footprint_mb(tts_wrapper.model)
    gpu_mem_after_load_mb = round(torch.cuda.memory_allocated() / _MB, 2) if torch.cuda.is_available() else 0.0

    load_meta = {
        **describe_quantization(llm_bit=llm_bit, mtp_bit=mtp_bit),
        "compute_dtype": dtype_to_str(compute_dtype),
        "attn_implementation": cfg.attn_implementation,
        "load_time_sec": round(time.perf_counter() - load_t0, 4),
        "model_snapshot_disk_size_mb": directory_size_mb(root_dir),
        "disk_size_mb": directory_size_mb(root_dir),
        "model_memory_footprint_mb": model_memory_footprint_mb,
        "gpu_mem_after_load_mb": gpu_mem_after_load_mb,
        "model_root_dir": str(root_dir),
    }
    return tts_wrapper, quant_report, load_meta

In [ ]:
# CELL 7: generation(생성) + inference(추론) 메트릭


# process ram
def get_process_ram_mb() -> float:
    try:
        return psutil.Process(os.getpid()).memory_info().rss / _MB
    except Exception:
        return float("nan")


def get_peak_gpu_mem_mb() -> float:
    if torch.cuda.is_available():
        try:
            return torch.cuda.max_memory_allocated() / _MB
        except Exception:
            return 0.0
    return 0.0


def get_allocated_gpu_mem_mb() -> float:
    if torch.cuda.is_available():
        try:
            return torch.cuda.memory_allocated() / _MB
        except Exception:
            return 0.0
    return 0.0


def get_reserved_gpu_mem_mb() -> float:
    if torch.cuda.is_available():
        try:
            return torch.cuda.memory_reserved() / _MB
        except Exception:
            return 0.0
    return 0.0


def guess_qwen_language(text: str) -> str:
    if re.search(r"[가-힣]", text or ""):
        return "Korean"
    if re.search(r"[\u3040-\u30ff\u31f0-\u31ff]", text or ""):
        return "Japanese"
    if re.search(r"[\u4e00-\u9fff]", text or ""):
        return "Chinese"
    return "Auto"


def load_audio_mono_24k(audio_path: str) -> tuple[np.ndarray, int]:
    wav, sr = sf.read(str(audio_path))
    wav = ensure_mono_float32(wav)
    if sr != 24000:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=24000)
        sr = 24000
    wav = np.clip(wav, -1.0, 1.0).astype(np.float32)
    return wav, sr


def build_batch_inputs(batch_rows: list[dict]) -> tuple[list[str], list[str], list[str], list[str]]:
    texts = [str(row["target_text"]) for row in batch_rows]
    languages = [guess_qwen_language(text) for text in texts]
    ref_audios = [str(row["ref_audio_path"]) for row in batch_rows]
    ref_texts = [str(row.get("ref_text", "") or "") for row in batch_rows]
    return texts, languages, ref_audios, ref_texts


def unpack_generation_result(result) -> tuple[list[np.ndarray], int]:
    if isinstance(result, (tuple, list)) and len(result) == 2:
        audio, sr = result
    else:
        audio = result
        sr = 24000

    if isinstance(audio, list):
        raw_wavs = audio
    elif torch.is_tensor(audio):
        arr = audio.detach().cpu().numpy()
        raw_wavs = [arr] if arr.ndim == 1 else [arr[i] for i in range(arr.shape[0])]
    else:
        arr = np.array(audio)
        raw_wavs = [arr] if arr.ndim == 1 else [arr[i] for i in range(arr.shape[0])]

    wavs: list[np.ndarray] = []
    for wav in raw_wavs:
        if torch.is_tensor(wav):
            wav = wav.detach().cpu().numpy()
        wav = ensure_mono_float32(np.array(wav).squeeze())
        wavs.append(wav)

    if not wavs:
        raise RuntimeError("생성된 오디오가 비어 있습니다.")

    return wavs, int(sr)


def generate_batch_via_high_level(
    cfg: ExperimentConfig,
    tts_wrapper,
    batch_rows: list[dict],
):
    texts, languages, ref_audios, ref_texts = build_batch_inputs(batch_rows)
    wavs, sr = tts_wrapper.generate_voice_clone(
        text=texts,
        language=languages,
        ref_audio=ref_audios,
        ref_text=ref_texts,
        non_streaming_mode=cfg.non_streaming_mode,
        max_new_tokens=cfg.max_new_tokens,
        do_sample=cfg.do_sample,
        top_k=cfg.top_k,
        top_p=cfg.top_p,
        temperature=cfg.temperature,
        subtalker_dosample=cfg.subtalker_dosample,
        use_cache=cfg.use_cache,
    )
    return wavs, sr


def generate_batch_via_prompt_fallback(
    cfg: ExperimentConfig,
    tts_wrapper,
    batch_rows: list[dict],
):
    texts, languages, _, ref_texts = build_batch_inputs(batch_rows)
    ref_audio_batch = []
    for row in batch_rows:
        ref_wav_24k, ref_sr_24k = load_audio_mono_24k(str(row["ref_audio_path"]))
        ref_audio_batch.append((ref_wav_24k, ref_sr_24k))

    prompt_items = tts_wrapper.create_voice_clone_prompt(
        ref_audio=ref_audio_batch,
        ref_text=ref_texts,
        x_vector_only_mode=[False] * len(batch_rows),
    )

    wavs, sr = tts_wrapper.generate_voice_clone(
        text=texts,
        language=languages,
        voice_clone_prompt=prompt_items,
        non_streaming_mode=cfg.non_streaming_mode,
        max_new_tokens=cfg.max_new_tokens,
        do_sample=cfg.do_sample,
        top_k=cfg.top_k,
        top_p=cfg.top_p,
        temperature=cfg.temperature,
        subtalker_dosample=cfg.subtalker_dosample,
        use_cache=cfg.use_cache,
    )
    return wavs, sr


def generate_batch_speech_with_metrics(
    cfg: ExperimentConfig,
    tts_wrapper,
    batch_rows: list[dict],
) -> tuple[list[np.ndarray], int, list[dict], dict]:
    if not batch_rows:
        raise ValueError("빈 batch_rows는 처리할 수 없습니다.")

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    ram_before_mb = get_process_ram_mb()
    t0 = time.perf_counter()

    try:
        result = generate_batch_via_high_level(
            cfg=cfg,
            tts_wrapper=tts_wrapper,
            batch_rows=batch_rows,
        )
        fallback_used = False
        batch_mode = "heterogeneous_prompt_batch_direct"
    except Exception as exc:
        print(f"  [batch fallback] direct batch generate_voice_clone 실패 ({type(exc).__name__}: {exc})")
        print("                   reference audio를 mono/24kHz로 정규화 후 batch prompt를 재생성합니다.")
        result = generate_batch_via_prompt_fallback(
            cfg=cfg,
            tts_wrapper=tts_wrapper,
            batch_rows=batch_rows,
        )
        fallback_used = True
        batch_mode = "heterogeneous_prompt_batch_prompt_fallback"

    batch_gen_sec = time.perf_counter() - t0
    peak_gpu_mb = get_peak_gpu_mem_mb()
    allocated_gpu_mb = get_allocated_gpu_mem_mb()
    reserved_gpu_mb = get_reserved_gpu_mem_mb()
    process_ram_mb = get_process_ram_mb()

    wavs, sr = unpack_generation_result(result)
    if len(wavs) != len(batch_rows):
        raise RuntimeError(
            f"배치 출력 개수와 입력 개수가 다릅니다. outputs={len(wavs)}, inputs={len(batch_rows)}"
        )

    batch_size_actual = len(batch_rows)
    amortized_gen_sec = batch_gen_sec / batch_size_actual if batch_size_actual > 0 else float("nan")
    audio_secs = [len(wav) / sr if sr else float("nan") for wav in wavs]
    batch_total_audio_sec = float(np.nansum(audio_secs)) if audio_secs else float("nan")
    batch_rtf = (
        batch_gen_sec / batch_total_audio_sec
        if batch_total_audio_sec and batch_total_audio_sec > 0
        else float("nan")
    )

    metrics_list: list[dict] = []
    for row, wav, audio_sec in zip(batch_rows, wavs, audio_secs):
        rtf = amortized_gen_sec / audio_sec if audio_sec and audio_sec > 0 else float("nan")
        metrics_list.append({
            "gen_sec": round(amortized_gen_sec, 4),
            "audio_sec": round(audio_sec, 4),
            "rtf": round(rtf, 4),
            "gen_latency_sec": round(amortized_gen_sec, 4),
            "gen_duration_sec": round(audio_sec, 4),
            "gen_rtf": round(rtf, 4),
            "batch_gen_sec": round(batch_gen_sec, 4),
            "batch_total_audio_sec": round(batch_total_audio_sec, 4),
            "batch_rtf": round(batch_rtf, 4),
            "batch_size_actual": int(batch_size_actual),
            "peak_gpu_mem_mb": round(peak_gpu_mb, 2),
            "allocated_gpu_mem_mb": round(allocated_gpu_mb, 2),
            "reserved_gpu_mem_mb": round(reserved_gpu_mb, 2),
            "process_ram_mb": round(process_ram_mb, 2),
            "process_ram_delta_mb": round(process_ram_mb - ram_before_mb, 2),
            "fallback_used": int(fallback_used),
            "generation_language": guess_qwen_language(str(row["target_text"])),
            "batch_mode": batch_mode,
        })

    batch_meta = {
        "batch_gen_sec": round(batch_gen_sec, 4),
        "batch_total_audio_sec": round(batch_total_audio_sec, 4),
        "batch_rtf": round(batch_rtf, 4),
        "batch_size_actual": int(batch_size_actual),
        "batch_mode": batch_mode,
    }
    return wavs, int(sr), metrics_list, batch_meta


def generate_speech_with_metrics(
    cfg: ExperimentConfig,
    tts_wrapper,
    target_text: str,
    ref_audio_path: str,
    ref_text: Optional[str] = None,
) -> tuple[np.ndarray, int, dict]:
    wavs, sr, metrics_list, _ = generate_batch_speech_with_metrics(
        cfg=cfg,
        tts_wrapper=tts_wrapper,
        batch_rows=[{
            "pair_id": "single",
            "target_text": target_text,
            "ref_audio_path": ref_audio_path,
            "ref_text": ref_text or "",
        }],
    )
    return wavs[0], int(sr), metrics_list[0]


In [ ]:
# CELL 8: 실험 실행
def load_manifest(cfg: ExperimentConfig) -> pd.DataFrame:
    if not cfg.reference_manifest_csv.exists():
        raise FileNotFoundError(f"manifest를 찾을 수 없습니다: {cfg.reference_manifest_csv}")
    manifest_df = pd.read_csv(cfg.reference_manifest_csv)
    required_cols = {"pair_id", "ref_audio_path", "target_text"}
    missing = required_cols - set(manifest_df.columns)
    if missing:
        raise ValueError(f"manifest 필수 컬럼 누락: {sorted(missing)}")
    return manifest_df


def print_config_summary(cfg: ExperimentConfig) -> None:
    print("=" * 80)
    print("실험 설정 요약")
    print("=" * 80)
    print(f"runtime_env    : {cfg.runtime_env}")
    print(f"project_root   : {cfg.project_root}")
    print(f"cache_dir      : {cfg.cache_dir}")
    print(f"manifest       : {cfg.reference_manifest_csv}")
    print(f"worker_tag     : {cfg.worker_tag}")
    print(f"worker_index   : {cfg.worker_index}")
    print(f"worker_count   : {cfg.worker_count}")
    print(f"target_device  : {cfg.target_device}")
    print(f"base_dtype     : {cfg.base_dtype_str}")
    print(f"attn_impl      : {cfg.attn_implementation}")
    print(f"batch_size     : {cfg.batch_size}")
    print(f"max_new_tokens : {cfg.max_new_tokens}")
    print(f"warmup_count   : {cfg.warmup_count}")
    print(f"sampling       : {cfg.sampling_config}")
    print(f"run plans      : {[plan.name for plan in cfg.run_plans]}")
    print()


def collect_env_info() -> dict[str, object]:
    info: dict[str, object] = {
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "pytorch_version": torch.__version__,
        "cuda_version": torch.version.cuda,
        "cudnn_version": torch.backends.cudnn.version(),
        "cpu_count_logical": psutil.cpu_count(logical=True),
        "cpu_count_physical": psutil.cpu_count(logical=False),
        "system_ram_mb": round(psutil.virtual_memory().total / _MB, 2),
    }
    if transformers is not None:
        info["transformers_version"] = getattr(transformers, "__version__", "unknown")
    if bnb is not None:
        info["bitsandbytes_version"] = getattr(bnb, "__version__", "unknown")

    if torch.cuda.is_available():
        gpu_idx = torch.cuda.current_device()
        props = torch.cuda.get_device_properties(gpu_idx)
        info.update({
            "gpu_name": props.name,
            "gpu_total_mem_mb": round(props.total_memory / _MB, 2),
            "gpu_capability": f"{props.major}.{props.minor}",
            "gpu_count": torch.cuda.device_count(),
            "bf16_supported": torch.cuda.is_bf16_supported(),
        })
    else:
        info.update({
            "gpu_name": "cpu_only",
            "gpu_total_mem_mb": 0.0,
            "gpu_capability": "none",
            "gpu_count": 0,
        })
    return info


def write_prompt_list(run_name: str, manifest_df: pd.DataFrame, cfg: ExperimentConfig) -> Path:
    prompt_path = cfg.result_dir / f"{run_name}_prompt_list.jsonl"
    if prompt_path.exists():
        return prompt_path

    with prompt_path.open("w", encoding="utf-8") as f:
        for _, row in manifest_df.iterrows():
            item = {
                "pair_id": str(row["pair_id"]),
                "target_text": str(row["target_text"]),
                "ref_audio_path": str(row["ref_audio_path"]),
                "ref_text": str(row.get("ref_text", "") or ""),
            }
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
    return prompt_path


def write_run_settings(
    cfg: ExperimentConfig,
    run_name: str,
    model_cfg,
    run_plan,
    run_order: int,
    manifest_df: pd.DataFrame,
    load_meta: dict[str, object],
    quant_report: dict[str, object],
    prompt_list_path: Path,
) -> Path:
    settings_path = cfg.result_dir / f"{run_name}_run_settings.json"
    env_info = collect_env_info()
    payload = {
        "saved_at": now_iso(),
        "run_name": run_name,
        "run_order": run_order,
        "model_id": model_cfg.model_id,
        "model_tag": model_cfg.model_tag,
        "manifest_path": str(cfg.reference_manifest_csv),
        "manifest_size": int(len(manifest_df)),
        "prompt_list_path": str(prompt_list_path),
        "seed": cfg.seed,
        "warmup_count": cfg.warmup_count,
        "sampling_config": cfg.sampling_config,
        "worker_tag": cfg.worker_tag,
        "worker_index": cfg.worker_index,
        "worker_count": cfg.worker_count,
        "runtime_env": cfg.runtime_env,
        "target_device": cfg.target_device,
        "base_dtype": cfg.base_dtype_str,
        "batch_size": cfg.batch_size,
        "quantization": {
            "llm_bit": run_plan.llm_bit,
            "mtp_bit": run_plan.mtp_bit,
            "keep_lm_head_in_base_dtype": cfg.keep_lm_head_in_base_dtype,
            "attn_implementation": cfg.attn_implementation,
            **load_meta,
            **quant_report,
        },
        "hardware": {
            "gpu_name": env_info.get("gpu_name"),
            "gpu_total_mem_mb": env_info.get("gpu_total_mem_mb"),
            "gpu_capability": env_info.get("gpu_capability"),
            "gpu_count": env_info.get("gpu_count"),
            "cpu_count_logical": env_info.get("cpu_count_logical"),
            "cpu_count_physical": env_info.get("cpu_count_physical"),
            "system_ram_mb": env_info.get("system_ram_mb"),
        },
        "software": {
            "python_version": env_info.get("python_version"),
            "platform": env_info.get("platform"),
            "pytorch_version": env_info.get("pytorch_version"),
            "transformers_version": env_info.get("transformers_version"),
            "bitsandbytes_version": env_info.get("bitsandbytes_version"),
            "cuda_version": env_info.get("cuda_version"),
            "cudnn_version": env_info.get("cudnn_version"),
        },
    }
    settings_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    return settings_path


def run_warmup(
    cfg: ExperimentConfig,
    tts_wrapper,
    manifest_df: pd.DataFrame,
    run_name: str,
) -> tuple[int, int]:
    if cfg.warmup_count <= 0 or manifest_df.empty:
        return 0, 0

    warmup_ok = 0
    for warmup_idx in range(cfg.warmup_count):
        row = manifest_df.iloc[warmup_idx % len(manifest_df)]
        try:
            generate_speech_with_metrics(
                cfg=cfg,
                tts_wrapper=tts_wrapper,
                target_text=str(row["target_text"]),
                ref_audio_path=str(row["ref_audio_path"]),
                ref_text=str(row.get("ref_text", "") or ""),
            )
            warmup_ok += 1
        except Exception:
            (cfg.log_dir / f"{run_name}_warmup_{warmup_idx:02d}_error.txt").write_text(
                traceback.format_exc(), encoding="utf-8"
            )
    cleanup_memory()
    return cfg.warmup_count, warmup_ok


def safe_round(value: float, digits: int = 4) -> float:
    try:
        if pd.isna(value):
            return float("nan")
        return round(float(value), digits)
    except Exception:
        return float("nan")


def percentile_or_nan(series: pd.Series, q: float) -> float:
    if series.empty:
        return float("nan")
    return float(series.quantile(q))


def iter_manifest_batches(records: list[dict], batch_size: int):
    if batch_size <= 0:
        raise ValueError(f"batch_size는 1 이상이어야 합니다. 받은 값: {batch_size}")
    for start in range(0, len(records), batch_size):
        yield records[start:start + batch_size]


def run_preflight_smoke(cfg: ExperimentConfig, manifest_df: pd.DataFrame) -> None:
    if not cfg.run_preflight_smoke or manifest_df.empty:
        print("[skip] smoke test 생략")
        return

    smoke_model_cfg = cfg.model_configs[0]
    smoke_run_plan = cfg.run_plans[0]
    print("\n=== Smoke Test ===")
    print(f"  model      : {smoke_model_cfg.model_tag}")
    print(f"  run_plan   : {smoke_run_plan.name}")

    smoke_row = manifest_df.iloc[0]
    print(f"  pair_id    : {smoke_row['pair_id']}")
    print(f"  ref_audio  : {smoke_row['ref_audio_path']}")
    print(f"  target_text: {smoke_row['target_text']}")

    smoke_out_dir = cfg.gen_dir / "_smoke"
    smoke_out_dir.mkdir(parents=True, exist_ok=True)
    smoke_out_path = smoke_out_dir / f"{smoke_row['pair_id']}.wav"

    tts_wrapper, quant_report, load_meta = load_model_for_run(
        cfg=cfg,
        model_id=smoke_model_cfg.model_id,
        llm_bit=smoke_run_plan.llm_bit,
        mtp_bit=smoke_run_plan.mtp_bit,
    )
    print("모델 로드 중...")
    print(f"  dtype={load_meta['compute_dtype']}  quant={quant_report}")

    audio, sr, metrics = generate_speech_with_metrics(
        cfg=cfg,
        tts_wrapper=tts_wrapper,
        target_text=str(smoke_row["target_text"]),
        ref_audio_path=str(smoke_row["ref_audio_path"]),
        ref_text=str(smoke_row.get("ref_text", "") or ""),
    )
    sf.write(str(smoke_out_path), audio, sr)
    print(f"smoke wav saved: {smoke_out_path}")
    print("smoke metrics :", metrics)

    del tts_wrapper
    cleanup_memory()


def run_experiment(cfg: ExperimentConfig) -> Path:
    print_config_summary(cfg)
    preload_model_snapshots(cfg)
    manifest_df = load_manifest(cfg)
    print(f"전체 pair 수: {len(manifest_df)}")
    print(f"manifest 컬럼: {manifest_df.columns.tolist()}")

    run_preflight_smoke(cfg, manifest_df)

    all_run_summaries: list[dict] = []
    experiment_start = time.time()
    run_order = 0

    for model_cfg in cfg.model_configs:
        print("\n" + "=" * 70)
        print(f"모델: {model_cfg.model_id} ({model_cfg.model_tag})")
        print("=" * 70)

        for run_plan in cfg.run_plans:
            run_order += 1
            run_name = f"{model_cfg.model_tag}_{run_plan.name}_{cfg.target_device}"
            run_gen_dir = cfg.gen_dir / run_name
            out_csv = cfg.result_dir / f"{run_name}_inference_manifest.csv"
            partial_csv = cfg.result_dir / f"{run_name}_inference_manifest.partial.csv"
            summary_csv = cfg.result_dir / f"{run_name}_run_summary.csv"

            if cfg.resume_incomplete_runs and out_csv.exists():
                print(f"  [SKIP] {run_name}")
                try:
                    prev = pd.read_csv(summary_csv).iloc[0].to_dict()
                    all_run_summaries.append(prev)
                except Exception:
                    pass
                continue

            run_gen_dir.mkdir(parents=True, exist_ok=True)
            prompt_list_path = write_prompt_list(run_name, manifest_df, cfg)

            print(f"\n  RUN: {run_name}")
            print(f"    llm_bit={run_plan.llm_bit}  mtp_bit={run_plan.mtp_bit}  batch_size={cfg.batch_size}")

            run_t0 = time.perf_counter()

            try:
                tts_wrapper, quant_report, load_meta = load_model_for_run(
                    cfg=cfg,
                    model_id=model_cfg.model_id,
                    llm_bit=run_plan.llm_bit,
                    mtp_bit=run_plan.mtp_bit,
                )
                print(f"    dtype={load_meta['compute_dtype']}  quant={quant_report}")
                print(f"    load_time={load_meta['load_time_sec']}s  model_mem={load_meta['model_memory_footprint_mb']} MB")
            except Exception as load_err:
                print(f"  [ERROR] 모델 로드 실패: {load_err}")
                (cfg.log_dir / f"{run_name}_load_error.txt").write_text(traceback.format_exc(), encoding='utf-8')
                continue

            settings_path = write_run_settings(
                cfg=cfg,
                run_name=run_name,
                model_cfg=model_cfg,
                run_plan=run_plan,
                run_order=run_order,
                manifest_df=manifest_df,
                load_meta=load_meta,
                quant_report=quant_report,
                prompt_list_path=prompt_list_path,
            )

            warmup_count_done, warmup_ok = run_warmup(
                cfg=cfg, tts_wrapper=tts_wrapper, manifest_df=manifest_df, run_name=run_name
            )

            done_pair_ids: set[str] = set()
            rows_out: list[dict] = []

            if cfg.resume_incomplete_runs and partial_csv.exists():
                prev_df = pd.read_csv(partial_csv)
                done_pair_ids = set(prev_df["pair_id"].astype(str).tolist())
                rows_out = prev_df.to_dict("records")
                print(f"    [Resume] partial: {len(done_pair_ids)}건 skip")

            pending_records = []
            for _, row in manifest_df.iterrows():
                pair_id = str(row["pair_id"])
                if cfg.resume_incomplete_runs and pair_id in done_pair_ids:
                    continue
                pending_records.append(row.to_dict())

            total_batches = (len(pending_records) + cfg.batch_size - 1) // cfg.batch_size if pending_records else 0

            for batch_index, batch_rows in enumerate(
                tqdm(iter_manifest_batches(pending_records, cfg.batch_size), total=total_batches, desc=f"  {run_plan.name}")
            ):
                batch_pair_ids = [str(row["pair_id"]) for row in batch_rows]
                batch_size_actual = len(batch_rows)

                common_run_meta = {
                    "run_name": run_name,
                    "run_order": run_order,
                    "model_id": model_cfg.model_id,
                    "model_tag": model_cfg.model_tag,
                    "quant_method": load_meta["quant_method"],
                    "bit_width": load_meta["bit_width"],
                    "llm_bit": run_plan.llm_bit,
                    "mtp_bit": run_plan.mtp_bit,
                    "llm_quant_method": load_meta["llm_quant_method"],
                    "mtp_quant_method": load_meta["mtp_quant_method"],
                    "compute_dtype": load_meta["compute_dtype"],
                    "seed": cfg.seed,
                    "warmup_count": cfg.warmup_count,
                    "manifest_size": int(len(manifest_df)),
                    "temperature": cfg.temperature,
                    "top_p": cfg.top_p,
                    "top_k": cfg.top_k,
                    "max_new_tokens": cfg.max_new_tokens,
                    "do_sample": int(cfg.do_sample),
                    "non_streaming_mode": int(cfg.non_streaming_mode),
                    "subtalker_dosample": int(cfg.subtalker_dosample),
                    "use_cache": int(cfg.use_cache),
                    "configured_batch_size": cfg.batch_size,
                    "batch_size_actual": batch_size_actual,
                    "batch_index": batch_index,
                    "load_time_sec": load_meta["load_time_sec"],
                    "model_memory_footprint_mb": load_meta["model_memory_footprint_mb"],
                    "disk_size_mb": load_meta["disk_size_mb"],
                    "model_snapshot_disk_size_mb": load_meta["model_snapshot_disk_size_mb"],
                    "gpu_mem_after_load_mb": load_meta["gpu_mem_after_load_mb"],
                    "prompt_list_path": str(prompt_list_path),
                    "run_settings_path": str(settings_path),
                }

                try:
                    for row in batch_rows:
                        if not Path(str(row["ref_audio_path"])).exists():
                            raise FileNotFoundError(f"ref_audio 없음: {row['ref_audio_path']}")

                    audios, sr, metrics_list, batch_meta = generate_batch_speech_with_metrics(
                        cfg=cfg,
                        tts_wrapper=tts_wrapper,
                        batch_rows=batch_rows,
                    )

                    if len(audios) != len(batch_rows):
                        raise RuntimeError(
                            f"생성된 오디오 수와 입력 수가 다릅니다. outputs={len(audios)}, inputs={len(batch_rows)}"
                        )

                    for batch_pos, (row, audio, metrics) in enumerate(zip(batch_rows, audios, metrics_list)):
                        pair_id = str(row["pair_id"])
                        out_path = run_gen_dir / f"{pair_id}.wav"
                        base_rec = dict(row)

                        sf.write(str(out_path), audio, sr)

                        record = {
                            **base_rec,
                            **common_run_meta,
                            **batch_meta,
                            "batch_pos_in_batch": batch_pos,
                            "gen_audio_path": str(out_path),
                            **metrics,
                            "status": "ok",
                            "error_msg": "",
                        }
                        rows_out.append(record)
                        done_pair_ids.add(pair_id)

                except Exception as batch_exc:
                    batch_err_path = cfg.log_dir / f"{run_name}_batch_{batch_index:04d}_error.txt"
                    batch_err_path.write_text(
                        f"batch_pair_ids={batch_pair_ids}\n\n{traceback.format_exc()}",
                        encoding="utf-8",
                    )
                    print(
                        f"    [batch->single fallback] batch_index={batch_index} "
                        f"pair_ids={batch_pair_ids}  err={type(batch_exc).__name__}: {batch_exc}"
                    )

                    for batch_pos, row in enumerate(batch_rows):
                        pair_id = str(row["pair_id"])
                        ref_audio = str(row["ref_audio_path"])
                        ref_text = str(row.get("ref_text", "") or "")
                        tgt_text = str(row["target_text"])
                        out_path = run_gen_dir / f"{pair_id}.wav"
                        base_rec = dict(row)

                        single_meta = {
                            **common_run_meta,
                            "batch_size_actual": 1,
                            "batch_pos_in_batch": batch_pos,
                            "batch_mode": "single_recovery",
                            "batch_gen_sec": float("nan"),
                            "batch_total_audio_sec": float("nan"),
                            "batch_rtf": float("nan"),
                        }

                        try:
                            if not Path(ref_audio).exists():
                                raise FileNotFoundError(f"ref_audio 없음: {ref_audio}")

                            audio, sr, metrics = generate_speech_with_metrics(
                                cfg=cfg,
                                tts_wrapper=tts_wrapper,
                                target_text=tgt_text,
                                ref_audio_path=ref_audio,
                                ref_text=ref_text or None,
                            )
                            sf.write(str(out_path), audio, sr)

                            record = {
                                **base_rec,
                                **single_meta,
                                "gen_audio_path": str(out_path),
                                **metrics,
                                "status": "ok",
                                "error_msg": "",
                            }
                        except Exception as exc:
                            record = {
                                **base_rec,
                                **single_meta,
                                "gen_audio_path": "",
                                "gen_sec": float("nan"),
                                "audio_sec": float("nan"),
                                "rtf": float("nan"),
                                "gen_latency_sec": float("nan"),
                                "gen_duration_sec": float("nan"),
                                "gen_rtf": float("nan"),
                                "peak_gpu_mem_mb": float("nan"),
                                "allocated_gpu_mem_mb": float("nan"),
                                "reserved_gpu_mem_mb": float("nan"),
                                "process_ram_mb": float("nan"),
                                "process_ram_delta_mb": float("nan"),
                                "fallback_used": float("nan"),
                                "generation_language": "",
                                "status": "error",
                                "error_msg": str(exc),
                            }
                            (cfg.log_dir / f"{run_name}_{pair_id}_error.txt").write_text(
                                traceback.format_exc(), encoding="utf-8"
                            )

                        rows_out.append(record)
                        done_pair_ids.add(pair_id)

                if len(rows_out) % cfg.write_partial_csv_every_n == 0:
                    pd.DataFrame(rows_out).to_csv(partial_csv, index=False, encoding="utf-8-sig")

            run_df = pd.DataFrame(rows_out)
            run_df.to_csv(out_csv, index=False, encoding="utf-8-sig")

            ok_df = run_df[run_df["status"] == "ok"].copy()
            n_ok = len(ok_df)
            n_error = int((run_df["status"] == "error").sum())
            n_total = len(run_df)
            success_rate = (n_ok / n_total) if n_total > 0 else float("nan")
            failure_rate = (n_error / n_total) if n_total > 0 else float("nan")

            env_info = collect_env_info()
            latency_series = ok_df["gen_sec"].dropna() if "gen_sec" in ok_df.columns else pd.Series(dtype=float)
            audio_series = ok_df["audio_sec"].dropna() if "audio_sec" in ok_df.columns else pd.Series(dtype=float)
            rtf_series = ok_df["rtf"].dropna() if "rtf" in ok_df.columns else pd.Series(dtype=float)
            batch_latency_series = ok_df["batch_gen_sec"].dropna() if "batch_gen_sec" in ok_df.columns else pd.Series(dtype=float)
            batch_rtf_series = ok_df["batch_rtf"].dropna() if "batch_rtf" in ok_df.columns else pd.Series(dtype=float)
            peak_gpu_series = ok_df["peak_gpu_mem_mb"].dropna() if "peak_gpu_mem_mb" in ok_df.columns else pd.Series(dtype=float)
            ram_series = ok_df["process_ram_mb"].dropna() if "process_ram_mb" in ok_df.columns else pd.Series(dtype=float)

            summary = {
                "run_name": run_name,
                "run_order": run_order,
                "saved_at": now_iso(),
                "model_id": model_cfg.model_id,
                "model_tag": model_cfg.model_tag,
                "quant_method": load_meta["quant_method"],
                "bit_width": load_meta["bit_width"],
                "llm_bit": run_plan.llm_bit,
                "mtp_bit": run_plan.mtp_bit,
                "llm_quant_method": load_meta["llm_quant_method"],
                "mtp_quant_method": load_meta["mtp_quant_method"],
                "compute_dtype": load_meta["compute_dtype"],
                "model_memory_footprint_mb": load_meta["model_memory_footprint_mb"],
                "disk_size_mb": load_meta["disk_size_mb"],
                "model_snapshot_disk_size_mb": load_meta["model_snapshot_disk_size_mb"],
                "gpu_mem_after_load_mb": load_meta["gpu_mem_after_load_mb"],
                "load_time_sec": load_meta["load_time_sec"],
                "n_total": n_total,
                "n_ok": n_ok,
                "n_error": n_error,
                "success_rate": safe_round(success_rate, 4),
                "failure_rate": safe_round(failure_rate, 4),
                "configured_batch_size": cfg.batch_size,
                "mean_batch_gen_sec": safe_round(batch_latency_series.mean(), 4),
                "median_batch_gen_sec": safe_round(batch_latency_series.median(), 4),
                "mean_batch_rtf": safe_round(batch_rtf_series.mean(), 4),
                "mean_gen_sec": safe_round(latency_series.mean(), 4),
                "median_latency_sec": safe_round(latency_series.median(), 4),
                "p95_latency_sec": safe_round(percentile_or_nan(latency_series, 0.95), 4),
                "mean_audio_sec": safe_round(audio_series.mean(), 4),
                "mean_rtf": safe_round(rtf_series.mean(), 4),
                "peak_gpu_mem_mb": safe_round(peak_gpu_series.max(), 2),
                "mean_peak_gpu_mem_mb": safe_round(peak_gpu_series.mean(), 2),
                "process_ram_mb": safe_round(ram_series.mean(), 2),
                "max_process_ram_mb": safe_round(ram_series.max(), 2),
                "warmup_count": cfg.warmup_count,
                "warmup_count_done": warmup_count_done,
                "warmup_success_count": warmup_ok,
                "seed": cfg.seed,
                "prompt_list_path": str(prompt_list_path),
                "manifest_path": str(cfg.reference_manifest_csv),
                "manifest_size": int(len(manifest_df)),
                "temperature": cfg.temperature,
                "top_p": cfg.top_p,
                "top_k": cfg.top_k,
                "max_new_tokens": cfg.max_new_tokens,
                "do_sample": int(cfg.do_sample),
                "non_streaming_mode": int(cfg.non_streaming_mode),
                "subtalker_dosample": int(cfg.subtalker_dosample),
                "use_cache": int(cfg.use_cache),
                "sampling_config_json": json.dumps(cfg.sampling_config, ensure_ascii=False),
                "worker_tag": cfg.worker_tag,
                "worker_index": cfg.worker_index,
                "worker_count": cfg.worker_count,
                "runtime_env": cfg.runtime_env,
                "target_device": cfg.target_device,
                "run_elapsed_sec": safe_round(time.perf_counter() - run_t0, 2),
                "settings_json_path": str(settings_path),
                "hardware_json": json.dumps({
                    "gpu_name": env_info.get("gpu_name"),
                    "gpu_total_mem_mb": env_info.get("gpu_total_mem_mb"),
                    "gpu_capability": env_info.get("gpu_capability"),
                    "gpu_count": env_info.get("gpu_count"),
                    "cpu_count_logical": env_info.get("cpu_count_logical"),
                    "cpu_count_physical": env_info.get("cpu_count_physical"),
                    "system_ram_mb": env_info.get("system_ram_mb"),
                }, ensure_ascii=False),
                "software_json": json.dumps({
                    "python_version": env_info.get("python_version"),
                    "platform": env_info.get("platform"),
                    "pytorch_version": env_info.get("pytorch_version"),
                    "transformers_version": env_info.get("transformers_version"),
                    "bitsandbytes_version": env_info.get("bitsandbytes_version"),
                    "cuda_version": env_info.get("cuda_version"),
                    "cudnn_version": env_info.get("cudnn_version"),
                }, ensure_ascii=False),
                "gpu_name": env_info.get("gpu_name"),
                "gpu_total_mem_mb": env_info.get("gpu_total_mem_mb"),
                "gpu_capability": env_info.get("gpu_capability"),
                "cpu_count_logical": env_info.get("cpu_count_logical"),
                "cpu_count_physical": env_info.get("cpu_count_physical"),
                "system_ram_mb": env_info.get("system_ram_mb"),
                "python_version": env_info.get("python_version"),
                "pytorch_version": env_info.get("pytorch_version"),
                "transformers_version": env_info.get("transformers_version"),
                "bitsandbytes_version": env_info.get("bitsandbytes_version"),
                "cuda_version": env_info.get("cuda_version"),
                "cudnn_version": env_info.get("cudnn_version"),
                "n_quantized": quant_report.get("n_quantized"),
                "n_skipped": quant_report.get("n_skipped"),
            }
            pd.DataFrame([summary]).to_csv(summary_csv, index=False, encoding="utf-8-sig")
            all_run_summaries.append(summary)

            print(f"\n    [완료] {run_name}")
            print(f"      success_rate          : {summary['success_rate']:.4f}")
            print(f"      Mean Batch Gen Sec    : {summary['mean_batch_gen_sec']:.3f}s")
            print(f"      Mean Batch RTF        : {summary['mean_batch_rtf']:.4f}")
            print(f"      Mean RTF              : {summary['mean_rtf']:.4f}")
            print(f"      Mean Latency          : {summary['mean_gen_sec']:.3f}s")
            print(f"      Median / P95 Latency  : {summary['median_latency_sec']:.3f}s / {summary['p95_latency_sec']:.3f}s")
            print(f"      Mean Audio Duration   : {summary['mean_audio_sec']:.3f}s")
            print(f"      Peak GPU Memory       : {summary['peak_gpu_mem_mb']:.1f} MB")
            print(f"      Mean Process RAM      : {summary['process_ram_mb']:.1f} MB")
            print(f"      Load Time             : {summary['load_time_sec']:.3f}s")

            del tts_wrapper
            cleanup_memory()

    total_elapsed = time.time() - experiment_start
    all_csv = cfg.result_dir / "ALL_RUNS_SUMMARY.csv"

    all_run_summaries_df = pd.DataFrame(all_run_summaries)
    all_run_summaries_df.to_csv(all_csv, index=False, encoding="utf-8-sig")

    print("\n" + "=" * 70)
    print(f"전체 실험 완료 ({total_elapsed / 60:.1f}분)")
    print(f"집계 CSV : {all_csv}")
    print("=" * 70)
    return all_csv


In [ ]:
# CELL 9: 실행 전 확인
print_config_summary(cfg)
print("reference_manifest_csv exists:", cfg.reference_manifest_csv.exists())
print("audio_ref_dir exists          :", cfg.audio_ref_dir.exists())
print("result_dir                    :", cfg.result_dir)
print("gen_dir                       :", cfg.gen_dir)

실험 설정 요약
runtime_env    : colab
project_root   : /content/drive/MyDrive/Colab Notebooks/Qwen3-TTS_Quantization_lowbit
cache_dir      : /content/hf_cache
manifest       : /content/drive/MyDrive/Colab Notebooks/Data/Qwen3-TTS_DATA/Manifests/reference_manifest.csv
worker_tag     : colab2
worker_index   : 2
worker_count   : 1
target_device  : cuda
base_dtype     : bfloat16
attn_impl      : flash_attention_2
batch_size     : 8
max_new_tokens : 512
warmup_count   : 1
sampling       : {'temperature': 1.0, 'top_p': 0.95, 'top_k': 50, 'max_new_tokens': 512, 'do_sample': True, 'non_streaming_mode': True, 'subtalker_dosample': False, 'use_cache': True, 'batch_size': 8}
run plans      : ['bf16_baseline', 'llm8_mtp8', 'llm4_mtp8', 'llm8_mtp4', 'llm4_mtp4', 'llm4_mtp3', 'llm4_mtp2', 'llm3_mtp4', 'llm2_mtp4', 'llm3_mtp3', 'llm2_mtp2']

reference_manifest_csv exists: True
audio_ref_dir exists          : True
result_dir                    : /content/drive/MyDrive/Colab Notebooks/Qwen3-TTS_Quantization_

In [ ]:
# CELL 10: 전체 실험 실행
all_csv_path = run_experiment(cfg)
print("saved:", all_csv_path)

실험 설정 요약
runtime_env    : colab
project_root   : /content/drive/MyDrive/Colab Notebooks/Qwen3-TTS_Quantization_lowbit
cache_dir      : /content/hf_cache
manifest       : /content/drive/MyDrive/Colab Notebooks/Data/Qwen3-TTS_DATA/Manifests/reference_manifest.csv
worker_tag     : colab2
worker_index   : 2
worker_count   : 1
target_device  : cuda
base_dtype     : bfloat16
attn_impl      : flash_attention_2
batch_size     : 8
max_new_tokens : 512
warmup_count   : 1
sampling       : {'temperature': 1.0, 'top_p': 0.95, 'top_k': 50, 'max_new_tokens': 512, 'do_sample': True, 'non_streaming_mode': True, 'subtalker_dosample': False, 'use_cache': True, 'batch_size': 8}
run plans      : ['bf16_baseline', 'llm8_mtp8', 'llm4_mtp8', 'llm8_mtp4', 'llm4_mtp4', 'llm4_mtp3', 'llm4_mtp2', 'llm3_mtp4', 'llm2_mtp4', 'llm3_mtp3', 'llm2_mtp2']


[preload] HF snapshot 캐시 확보 중...
  Qwen/Qwen3-TTS-12Hz-1.7B-Base


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/127 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

[WARNING] Min value of input waveform signal is -1.0063717365264893
[WARNING] Max value of input waveform signal is 1.0085132122039795
[WARNING] Min value of input waveform signal is -1.001962661743164
[WARNING] Min value of input waveform signal is -1.001097559928894
[WARNING] Max value of input waveform signal is 1.0015896558761597
[WARNING] Min value of input waveform signal is -1.0012803077697754
[WARNING] Min value of input waveform signal is -1.0003712177276611
[WARNING] Min value of input waveform signal is -1.0105454921722412
[WARNING] Min value of input waveform signal is -1.0025765895843506
[WARNING] Min value of input waveform signal is -1.0016816854476929


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


[WARNING] Min value of input waveform signal is -1.000845193862915
[WARNING] Min value of input waveform signal is -1.0034360885620117
[WARNING] Max value of input waveform signal is 1.001955270767212
[WARNING] Min value of input waveform signal is -1.0024324655532837
[WARNING] Max value of input waveform signal is 1.0013920068740845
[WARNING] Min value of input waveform signal is -1.0043373107910156
[WARNING] Min value of input waveform signal is -1.0019114017486572
[WARNING] Max value of input waveform signal is 1.0009373426437378
[WARNING] Min value of input waveform signal is -1.0019524097442627
[WARNING] Max value of input waveform signal is 1.0038844347000122
[WARNING] Min value of input waveform signal is -1.0143345594406128
[WARNING] Max value of input waveform signal is 1.0009710788726807
[WARNING] Min value of input waveform signal is -1.000694990158081
[WARNING] Max value of input waveform signal is 1.0024640560150146


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.



    [완료] 1.7B_llm2_mtp2_cuda
      success_rate          : 1.0000
      Mean Batch Gen Sec    : 472.156s
      Mean Batch RTF        : 5.2558
      Mean RTF              : 58.6503
      Mean Latency          : 59.681s
      Median / P95 Latency  : 61.516s / 66.054s
      Mean Audio Duration   : 13.719s
      Peak GPU Memory       : 2953.9 MB
      Mean Process RAM      : 6835.0 MB
      Load Time             : 7.178s

전체 실험 완료 (1124.0분)
집계 CSV : /content/drive/MyDrive/Colab Notebooks/Qwen3-TTS_Quantization_lowbit/results/colab2/ALL_RUNS_SUMMARY.csv
saved: /content/drive/MyDrive/Colab Notebooks/Qwen3-TTS_Quantization_lowbit/results/colab2/ALL_RUNS_SUMMARY.csv
